In [15]:
import numpy as np

# Ввод матрицы платежей
C = np.array([[12, 9, 18], 
              [15, 22, 5], 
              [16, 3, 12]])

# Метод обратной матрицы (аналитическое решение)
# Вычисляем определитель и обратную матрицу.
# Используем единичный вектор u = [1, 1, 1].
# Находим оптимальные стратегии и цену игры.

# Функция для вычисления определителя матрицы
def determinant_matrix(mat):
    n = len(mat) # размерность квадратной матрицы
    if n == 1:
        return mat[0][0]
    if n == 2:
        return mat[0][0] * mat[1][1] - mat[0][1] * mat[1][0]
    
    det = 0
    for j in range(n): # перебираем элементы первой строки
        sub_mat = np.delete(np.delete(mat, 0, axis=0), j, axis=1) # минор (подматрица, которая остаётся после удаления i-й строки и j-го столбца)
        det += ((-1) ** j) * mat[0][j] * determinant_matrix(sub_mat) # ищем определитель: (-1)^j * c_0j * det(sub_mat)

    return det

# Функция для вычисления обратной матрицы
def inverse_matrix(mat):
    n = len(mat)
    det = determinant_matrix(mat)

    if det == 0:
        raise ValueError("Матрица вырожденная, обратной матрицы не существует.") # генерация исключения
    
    # Создаём матрицу алгебраических дополнений
    adjugate = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            sub_mat = np.delete(np.delete(mat, i, axis=0), j, axis=1) 
            adjugate[j][i] = ((-1) ** (i + j)) * determinant_matrix(sub_mat)
    
    return adjugate / det

C_inv = inverse_matrix(C) #обратная матрица
# Аналитический метод (обратная матрица)
# Вычисление оптимальных стратегий
u = np.ones(len(C))
v = 1 / np.dot(np.dot(u, C_inv), u.T) 
x_opt = np.dot(u, C_inv) * v
y_opt = np.dot(C_inv, u.T) * v

In [14]:
# Метод Брауна–Робинсон (итерационный алгоритм)
def braun_robinson(C, epsilon):
    n, m = C.shape # число стратегий игроков

    A = np.zeros(n) # массив из n нулей
    B = np.zeros(m) # массив из m нулей

    x_count = np.zeros(n)
    y_count = np.zeros(m)

    A = C[:,0] # первый столбец C
    B = C[0,:] # первая строка C

    x_count[0] += 1
    y_count[0] += 1
    
    I_A = 0
    I_B = 0

    v_up_arr = []
    v_down_arr= []

    eps = 1000
    k = 0

    headers = ["k", "Выбор A", "Выбор B", "x1", "x2", "x3", "y1", "y2", "y3", "1/k*u_up[k]", "1/k*u_down[k]", "epsilon"]
    print(f"{headers[0]:<3} {headers[1]:<10} {headers[2]:<10} {headers[3]:<5} {headers[4]:<5} {headers[5]:<5} {headers[6]:<5} {headers[7]:<5} {headers[8]:<5} {headers[9]:<12} {headers[10]:<12} {headers[11]:<5}")
    print("-" * 90)  
    
    while eps >= epsilon:
        k+=1 # номер итерации
        x_count[I_A] += 1 # увеличение числа выборов одной из стратегий игрока А
        y_count[I_B] += 1 # увеличение числа выборов одной из стратегий игрока B
        v_up   = max(A)/k # верхняя цена игры
        v_down = min(B)/k # нижняя цена игры
        v_up_arr.append(v_up) # массив верхних цен
        v_down_arr.append(v_down) # массив нижних цен

        eps = min(v_up_arr) - max(v_down_arr) # погрешность 

        print(f"{k:<3} x_{I_A+1:<10} y_{I_B+1:<6} {A[0]:<5} {A[1]:<5} {A[2]:<5} {B[0]:<5} {B[1]:<5} {B[2]:<5} {round(v_up,3):<12} {round(v_down,3):<12} {round(eps,3):<5}")
        print("-" * 90)  
        I_A = np.argmax(A) # индекс максимального элемента стратегии 
        I_B = np.argmin(B) # индекс минимального элемента стратегии
        A = A+C[:,I_B]
        B = B+C[I_A,:]

    x_opt_br = x_count/k
    y_opt_br = y_count/k
    print("Метод обратной матрицы\n") 
    print(f"Цена игры (v): {v}") 
    print(f"Оптимальная стратегия игрока A: {x_opt}")
    print(f"Оптимальная стратегия игрока B: {y_opt}")
    print(f"\n")

    print("Алгоритм Брауна-Робинсон\n")
    print(f"Оптимальная стратегия игрока A: {x_opt_br}\n")
    print(f"Оптимальная стратегия игрока B: {y_opt_br}\n")
    print(f"Погрешность: eps[{k}] = {round(eps,3)}")
    
braun_robinson(C,epsilon=0.1)



k   Выбор A    Выбор B    x1    x2    x3    y1    y2    y3    1/k*u_up[k]  1/k*u_down[k] epsilon
------------------------------------------------------------------------------------------
1   x_1          y_1      12    15    16    12    9     18    16.0         9.0          7.0  
------------------------------------------------------------------------------------------
2   x_3          y_2      21    37    19    28    12    30    18.5         6.0          7.0  
------------------------------------------------------------------------------------------
3   x_2          y_2      30    59    22    43    34    35    19.667       11.333       4.667
------------------------------------------------------------------------------------------
4   x_2          y_2      39    81    25    58    56    40    20.25        10.0         4.667
------------------------------------------------------------------------------------------
5   x_2          y_3      57    86    37    73    78    45    17.2      